# Import Basic Libraries and Modules

In [ ]:
# Loading Libraries
import warnings
warnings.filterwarnings('ignore')

import gc
gc.collect()

import pandas as pd
import numpy as np
import pickle
import os
import time

from rdkit.Chem import Descriptors
from rdkit.Chem import AllChem, Descriptors3D

from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, cohen_kappa_score

import shap

print("FInished loading libraries", flush=True)

# Get RDKit Descriptor Names

In [ ]:
RDKit_Descriptor_names = [name for name, _ in Descriptors._descList if name != "Ipc"]
RDKit_Descriptor_names= RDKit_Descriptor_names[:-1]

RDKit_Descriptor_names_all = []
for i in range(1, 5):  # For constituents 1 to 4
    RDKit_Descriptor_names_all.extend([f"{name} {i}" for name in RDKit_Descriptor_names])


# Add extra descriptors
extra_descriptors = ['Composition 1', 'Composition 2', 'Composition 3', 
                     'Composition 4', 'RNA type', 'Lipid to RNA', 'Dosage']
RDKit_Descriptor_names_all.extend(extra_descriptors)

print(len(RDKit_Descriptor_names_all), flush=True)


RDKit_3DDescriptor_names= ['Asphericity', 'Eccentricity', 'InertialShapeFactor', 'NPR1', 'NPR2', 
                           'PBF', 'PMI1', 'PMI2', 'PMI3', 'RadiusOfGyration', 'SpherocityIndex']

RDKit_3DDescriptor_names_all = []
for i in range(1, 5):  # For constituents 1 to 4
    RDKit_3DDescriptor_names_all.extend([f"{name} {i}" for name in RDKit_3DDescriptor_names])


# Add extra descriptors
extra_descriptors = ['Composition 1', 'Composition 2', 'Composition 3', 
                     'Composition 4', 'RNA type', 'Lipid to RNA', 'Dosage']
RDKit_3DDescriptor_names_all.extend(extra_descriptors)

print(len(RDKit_3DDescriptor_names_all), flush=True)



# Inputs & Parameters

In [ ]:

# File paths and other parameters
# File paths and other parameters
Nbins= 2
property= 'Activity'
target_style= 'NG'
featurizer_style= 'All_Featurizers_'+target_style
ml_model_style= 'All_Models_'+target_style
importance_style= 'SHAP'
featurizer_name= 'RDKit_Descriptors_NG'
feature_names = RDKit_Descriptor_names_all
sheet_name= featurizer_name


dataX_filepath = f"/home/kumar542/SAR_NM/Codes/ML_Framework/Data_Combined/{property}_pickle_files/dataX_dict_all_{featurizer_style}.pkl"
datay_filepath = f"/home/kumar542/SAR_NM/Codes/ML_Framework/Data_Combined/{property}_pickle_files/datay_all_{Nbins}bins_{target_style}.pkl"
scaffolds_filepath = rf"/home/kumar542/SAR_NM/Codes/ML_Framework/Data_Combined/{property}_pickle_files/Scaffolds.pkl"


interactions_file= f"/home/kumar542/SAR_NM/Codes/ML_Framework/Data_Combined/Feature_Importance/{property}_pickle_files/{property}_{Nbins} bins_{importance_style}_Interactions_{featurizer_name}.pkl"

interactions_output_file = f"/home/kumar542/SAR_NM/Codes/ML_Framework/Data_Combined/Feature_Importance/{property}_{Nbins} bins_{importance_style}_Interactions.xlsx"



# Implementation >>>>

# Load pickle files

In [ ]:
# Load datasets

with open(dataX_filepath, 'rb') as file:
    print('Reading feature variable')
    dataX_dict_all = pickle.load(file)
    
with open(datay_filepath, 'rb') as file1:
    print('Reading target variable')
    datay_all= pickle.load(file1)

with open(scaffolds_filepath, 'rb') as file1:
    print('Reading scaffolds file')
    scaffolds= pickle.load(file1)

dataX_all= dataX_dict_all[featurizer_name]

num_features = dataX_all.shape[1]
feature_id = [i for i in range(num_features)]
print(f"Number of features is featurizer is {num_features}", flush=True)


print("Finished loading datasets", flush=True)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold_idx, (train_idx, test_idx) in enumerate(kf.split(scaffolds)):
     if fold_idx==0:
         datay_train, datay_test = datay_all[train_idx], datay_all[test_idx]
         features_train, features_test = dataX_all[train_idx], dataX_all[test_idx]


  


# SHAP Importance

In [ ]:
print("Starting SHAP Importance Calculation", flush=True)

model = RandomForestClassifier(n_estimators= 100, max_depth= 10, min_samples_leaf= 4, min_samples_split= 10, random_state= 42, class_weight= 'balanced')
model.fit(features_train, datay_train)

explainer = shap.TreeExplainer(model)
shap_values_all = explainer.shap_values(features_test)
shap_values= shap_values_all[1]

# Compute average SHAP values (absolute importance)
shap_importance = np.abs(shap_values).mean(axis=0)
shap_importance_df = pd.DataFrame({'Feature ID': feature_id, 'Feature Name': feature_names, 'SHAP Importance': shap_importance})
shap_importance_sorted = shap_importance_df.sort_values(by='SHAP Importance', ascending=False)

print(f"Finished SHAP Importance Score", flush=True)



# SHAP Interactions

In [ ]:
print("Starting SHAP Interactions Calculation", flush=True)

# Compute SHAP interaction values only for the selected features
interaction_values_all = explainer.shap_interaction_values(features_test)

# Keep only interactions for one class (e.g., class 1)
interaction_values = interaction_values_all[1]  # This reduces (n_samples, top_k, top_k, 2) to (n_samples, top_k, top_k)

with open(interactions_file, 'wb') as file1:
    print('Writing Interactions pickle file', flush=True)
    pickle.dump(interaction_values, file1)

interaction_list = []
for i in range(num_features):
    for j in range(i+1, num_features):  # Avoid double counting
        interaction_importance = np.abs(interaction_values[:, i, j]).mean()
        interaction_list.append((feature_names[i], feature_names[j], interaction_importance))

interaction_df = pd.DataFrame(interaction_list, columns=['Feature 1', 'Feature 2', 'Interaction Importance'])
interaction_df_sorted = interaction_df.sort_values(by='Interaction Importance', ascending=False)


# Save to Excel
with pd.ExcelWriter(interactions_output_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:    
    interaction_df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Finished computing interactions .", flush=True)


In [4]:
# !jupyter nbconvert --to script SHAP_Importance_HPC_Descriptors.ipynb

[NbConvertApp] Converting notebook SHAP_Importance_HPC_Descriptors.ipynb to script
[NbConvertApp] Writing 6319 bytes to SHAP_Importance_HPC_Descriptors.py
